In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP01 - TP Jurisdictions
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Country Risk Ratings: hive_metastore.ra_adido_2025.td_country_risk_rating_abac
   4. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   What number of third party relationships with third parties are in jurisdictions 
   considered a high risk for bribery and corruption during the assessment period?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the new Master AU List + TP Unit Mapping, strictly 
      filtered by 'Jurisdiction' for Canada, Hong Kong, and Barbados.
   2. HIGH RISK EVALUATION: Checks three columns against the ABAC High Risk list. 
      If ALL THREE are blank/null, automatically defaults to High Risk.
   3. TPRM STRING MAPPING (FIXED): Maps TP engagements to AU IDs by checking if the 
      'TPRM_Units' string exists inside the 'owning_lob' or 'lob_using' columns.
      - Blocks blank mapping strings using NULLIF to prevent wildcard explosions.
      - Uses Databricks RLIKE with word boundaries (\b) for exact phrase matching.
   4. AGGREGATING: Counts DISTINCT EngagementNumbers per mapped AU to remove duplicates.
   5. FINAL OUTPUT: Strict 6-column structure, converting NULL sums to '0'.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data and strictly filter by target jurisdictions
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment 
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
),

Mapped_AUs AS (
    -- 2. Grab every AU that currently has TPRM units mapped to it
    SELECT DISTINCT TRIM(CAST(Assessable_Unit_ID AS STRING)) AS AU_ID 
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
),

All_Base_AUs AS (
    -- 3. Combine filtered Master List and Mapping to create the ultimate base table
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        mast.Business_Segment,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

High_Risk_Countries AS (
    -- 4. Grab high-risk jurisdictions from the ABAC rating table
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Third_Party_Risk_Data AS (
    -- 5. Extract TP data and flag if jurisdiction is completely missing
    SELECT 
        EngagementNumber,
        owning_lob,
        lob_using,
        location_of_tp,
        country_prod_serv_originates,
        Td_Countries_Impacted,
        -- "If no Juristiction, then automatically high risk"
        CASE WHEN COALESCE(TRIM(location_of_tp), '') = ''
              AND COALESCE(TRIM(country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(Td_Countries_Impacted), '') = '' THEN 1
             ELSE 0 END AS Is_Missing_Jurisdiction
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

High_Risk_Engagements AS (
    -- 6. Filter for engagements that touch a High Risk country or have NO country
    SELECT DISTINCT 
        tp.EngagementNumber,
        tp.owning_lob,
        tp.lob_using
    FROM Third_Party_Risk_Data tp
    LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(tp.location_of_tp)) = h1.CountryName
    LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(tp.country_prod_serv_originates)) = h2.CountryName
    LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(tp.Td_Countries_Impacted)) = h3.CountryName
    WHERE h1.CountryName IS NOT NULL
       OR h2.CountryName IS NOT NULL
       OR h3.CountryName IS NOT NULL
       OR tp.Is_Missing_Jurisdiction = 1
),

Engagements_By_AU AS (
    -- 7. Map high-risk engagements using the new TPRM unit mapping table
    SELECT 
        TRIM(CAST(map.Assessable_Unit_ID AS STRING)) AS Mapped_AU_ID,
        COUNT(DISTINCT hr.EngagementNumber) AS High_Risk_Count
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping map
    
    -- FIXED JOIN: Uses Regex word boundaries and completely blocks blank mapping strings
    INNER JOIN High_Risk_Engagements hr
        ON NULLIF(TRIM(map.TPRM_Units), '') IS NOT NULL
       AND (
           UPPER(hr.owning_lob) RLIKE '\\b' || UPPER(TRIM(map.TPRM_Units)) || '\\b'
           OR UPPER(hr.lob_using) RLIKE '\\b' || UPPER(TRIM(map.TPRM_Units)) || '\\b'
       )
    WHERE map.Assessable_Unit_ID IS NOT NULL
    GROUP BY TRIM(CAST(map.Assessable_Unit_ID AS STRING))
)

-- 8. Final Template: Strict 6-column output
SELECT 
    a.Base_AU_ID AS BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP01' AS QuestionID,               
    COALESCE(CAST(e.High_Risk_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM All_Base_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.Base_AU_ID = e.Mapped_AU_ID
ORDER BY a.Base_AU_ID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP01 - TP Jurisdictions Drill-Down
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Country Risk Ratings: hive_metastore.ra_adido_2025.td_country_risk_rating_abac
   4. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   PURPOSE: 
   Provides a row-by-row view of every High Risk Third Party Engagement mapped to an AU.
   Displays the raw 'lob_using' and 'owning_lob' strings next to the mapped 'TPRM_Units'
   string to verify the text mapping logic, and explains the Risk flag reason. Uses 
   strict Regex bounds to prevent false positive matches.
=================================================================================== */

WITH Master_AUs AS (
    -- Grab Master List data and strictly filter by target jurisdictions
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
),

Mapped_AUs AS (
    -- Grab every AU that currently has TPRM units mapped to it
    SELECT DISTINCT TRIM(CAST(Assessable_Unit_ID AS STRING)) AS AU_ID 
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
),

All_Base_AUs AS (
    -- Combine filtered Master List and Mapping to create the ultimate base table
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

High_Risk_Countries AS (
    -- Grab high-risk jurisdictions from the ABAC rating table
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Third_Party_Risk_Data AS (
    -- Extract TP data and flag if jurisdiction is completely missing
    SELECT 
        EngagementNumber,
        ThirdPartyName,
        owning_lob,
        lob_using,
        location_of_tp,
        country_prod_serv_originates,
        Td_Countries_Impacted,
        CASE WHEN COALESCE(TRIM(location_of_tp), '') = ''
              AND COALESCE(TRIM(country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(Td_Countries_Impacted), '') = '' THEN 1
             ELSE 0 END AS Is_Missing_Jurisdiction
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
)

SELECT DISTINCT
    a.Base_AU_ID AS BusinessID,
    a.AU_Name,
    a.In_ABAC_AU_List,
    tp.EngagementNumber,
    tp.ThirdPartyName,
    map.TPRM_Units AS Successfully_Mapped_String,
    tp.owning_lob AS Raw_Col_K_owning_lob,
    tp.lob_using AS Raw_Col_L_lob_using,
    tp.location_of_tp AS Country_1,
    tp.country_prod_serv_originates AS Country_2,
    tp.Td_Countries_Impacted AS Country_3,
    
    CASE 
        WHEN tp.Is_Missing_Jurisdiction = 1 THEN 'Missing Jurisdiction (Auto High Risk)'
        ELSE 'Matched High Risk ABAC List' 
    END AS Risk_Flag_Reason
    
FROM All_Base_AUs a

-- Inner join AU list to mapping table
INNER JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map
    ON a.Base_AU_ID = TRIM(CAST(map.Assessable_Unit_ID AS STRING))

-- FIXED JOIN: Uses Regex word boundaries and completely blocks blank mapping strings
INNER JOIN Third_Party_Risk_Data tp
    ON NULLIF(TRIM(map.TPRM_Units), '') IS NOT NULL
   AND (
       UPPER(tp.owning_lob) RLIKE '\\b' || UPPER(TRIM(map.TPRM_Units)) || '\\b'
       OR UPPER(tp.lob_using) RLIKE '\\b' || UPPER(TRIM(map.TPRM_Units)) || '\\b'
   )
    
LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(tp.location_of_tp)) = h1.CountryName
LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(tp.country_prod_serv_originates)) = h2.CountryName
LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(tp.Td_Countries_Impacted)) = h3.CountryName

-- Strictly filter for only the high-risk engagements
WHERE h1.CountryName IS NOT NULL
   OR h2.CountryName IS NOT NULL
   OR h3.CountryName IS NOT NULL
   OR tp.Is_Missing_Jurisdiction = 1
   
ORDER BY a.Base_AU_ID;

In [ ]:
%sql
/* DIAGNOSTIC 1: String Mapping Test */
SELECT 
    tp.EngagementNumber, 
    tp.owning_lob, 
    tp.lob_using, 
    map.TPRM_Units AS Attempted_Match_String, 
    map.Assessable_Unit_ID
FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 tp
-- Try to join using the fuzzy text match
INNER JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map
    ON UPPER(tp.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
    OR UPPER(tp.lob_using) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
LIMIT 100;

In [ ]:
%sql
/* DIAGNOSTIC 2: High Risk Country spelling test */
WITH High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
)
SELECT 
    tp.EngagementNumber, 
    tp.location_of_tp AS Raw_TP_Country, 
    h.CountryName AS Matched_ABAC_Country
FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 tp
LEFT JOIN High_Risk_Countries h 
    ON UPPER(TRIM(tp.location_of_tp)) = h.CountryName
-- Only show rows where a country was actually listed
WHERE tp.location_of_tp IS NOT NULL
LIMIT 100;

In [ ]:
%sql
/* DIAGNOSTIC 3: Master AU Filter Test */
SELECT 
    BusinessID, 
    Business_Operational_Unit_Name, 
    Jurisdiction
FROM hive_metastore.ra_adido_2025.fy25_list_of_units
WHERE UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS');